*Hello There*

In [13]:
import numpy as np
import pandas as pd
import pypsa

In [18]:
x = np.sin(3.14)
print(x)

0.0015926529164868282


In [28]:
df = pd.read_csv("../data/ammonia_plants.csv")
df

,Plant,Ammonia [kt/a],Urea [kt/a],AN [kt/a],UAN [kt/a],CAN [kt/a],Nitrogen fertilisers [kt/a],MAP [kt/a],Country,Latitude,Longitude,Date,Source,Comment
0,Yara Sluiskil,1700.0,1300.0,NaN,NaN,NaN,NaN,NaN,Netherlands,51.27186,3.84896,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
1,OCI Geleen,550.0,NaN,NaN,NaN,NaN,NaN,NaN,Netherlands,50.99738,5.78497,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
2,CF Fertilisers Billingham,590.0,NaN,NaN,NaN,NaN,NaN,NaN,United Kingdom,53.28149,-2.79744,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
3,Yara Tertre,420.0,NaN,NaN,NaN,NaN,NaN,NaN,Belgium,50.47842,3.80285,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
4,BASF Antwerp,650.0,NaN,NaN,NaN,NaN,NaN,NaN,Belgium,51.35583,4.27811,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
5,Yara Gonfreville,400.0,NaN,NaN,NaN,NaN,NaN,NaN,France,49.47982,0.19587,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
6,Borealis Rouen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,France,49.41916,1.02584,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
7,BASF Ludwigshafen,880.0,NaN,NaN,NaN,NaN,NaN,NaN,Germany,49.51171,8.42009,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
8,SKW,1300.0,NaN,NaN,NaN,NaN,NaN,NaN,Germany,51.87746,12.58580,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN
9,Yara Brunsbuettel,750.0,NaN,NaN,NaN,NaN,NaN,NaN,Germany,53.91152,9.20982,13-01-2023,https://www.icis.com/explore/resources/news/20...,NaN


In [34]:
# marginal costs in EUR/MWh
marginal_costs = pd.DataFrame({
    "A": {"Coal": 35, "RES": 1, "Gas": 80},
    "B": {"Gas": 70, "Hydro": 2},
    "C": {"RES": 0},
})
marginal_costs.columns.name = "region"
marginal_costs.index.name = "technology"

# power plant capacities (nominal powers in MW) in each country
capacities = pd.DataFrame({
    "A": {"Coal": 8000, "RES": 1000, "Gas": 4000},
    "B": {"Gas": 2000, "Hydro": 4000},
    "C": {"RES": 3000},
})
capacities.columns.name = "region"
capacities.index.name = "technology"

# transmission capacities in MW
transmission = pd.Series({"A-B": 2000, "A-C": 4000, "B-C": 300})
transmission.index.name = "transmission"

# country electrical loads in MW
loads = pd.Series({"A": 10000, "B": 4000, "C": 1000}, name="region")

In [36]:
capacities_full = capacities.fillna(0)
marginal_costs_full = marginal_costs.fillna(0)

countries = list(capacities_full.columns)
technologies = list(capacities_full.index)

region_index = pd.Index(countries, name="region")
tech_index = pd.Index(technologies, name="technology")
line_index = pd.Index(transmission.index, name="transmission")

In [38]:
n = pypsa.Network()

# Tell Pandas to use standard strings to prevent the Xarray compatibility bug
pd.options.future.infer_string = False
pd.options.mode.string_storage = "python"

for country in countries:
    n.add("Bus", country)

for country in countries:
    for tech, p_nom in capacities[country].items():
        if pd.isna(p_nom):
            continue

        n.add(
            "Generator",
            f"{tech} {country}",
            bus=country,
            carrier=tech,
            p_nom=p_nom,
            marginal_cost=marginal_costs.loc[tech, country],
        )

for country in countries:
    n.add(
        "Load",
        f"{country} Demand",
        bus=country,
        p_set=loads[country],
        carrier="AC",
    )

for line_name, s_nom in transmission.items():
    bus0, bus1 = line_name.split("-")

    n.add(
        "Line",
        line_name,
        bus0=bus0,
        bus1=bus1,
        s_nom=s_nom,
        x=1,
        r=1,
    )
    
n.optimize(solver_name="gurobi", log_to_console=False)

C:\Users\User\AppData\Local\Temp\ipykernel_19468\1250315664.py:46: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize(solver_name="gurobi", log_to_console=False)
Index(['A', 'B', 'C'], dtype='object', name='name')
Index(['Coal A', 'RES A', 'Gas A', 'Gas B', 'Hydro B', 'RES C'], dtype='object', name='name')
Index(['A Demand', 'B Demand', 'C Demand'], dtype='object', name='name')
Index(['A-B', 'A-C', 'B-C'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.31s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2776863


INFO:gurobipy:Set parameter LicenseID to value 2776863


...

obj: 22 rows, 9 columns, 33 nonzeros


INFO:gurobipy:obj: 22 rows, 9 columns, 33 nonzeros


Set parameter LogToConsole to value 0


INFO:gurobipy:Set parameter LogToConsole to value 0
INFO:gurobipy:Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))
INFO:gurobipy:
INFO:gurobipy:CPU model: AMD Ryzen 5 PRO 4650U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
INFO:gurobipy:Thread count: 6 physical cores, 12 logical processors, using up to 12 threads
INFO:gurobipy:
INFO:gurobipy:Non-default parameters:
INFO:gurobipy:LogToConsole  0
INFO:gurobipy:
INFO:gurobipy:Optimize a model with 22 rows, 9 columns and 33 nonzeros (Min)
INFO:gurobipy:Model fingerprint: 0x45c2913c
INFO:gurobipy:Model has 5 linear objective coefficients
INFO:gurobipy:Coefficient statistics:
INFO:gurobipy:  Matrix range     [1e+00, 1e+05]
INFO:gurobipy:  Objective range  [1e+00, 8e+01]
INFO:gurobipy:  Bounds range     [0e+00, 0e+00]
INFO:gurobipy:  RHS range        [3e+02, 1e+04]
INFO:gurobipy:
INFO:gurobipy:Presolve removed 19 rows and 2 columns
INFO:gurobipy:Presolve time: 0.03s
INFO:gurobipy:Presolved: 3 rows, 7 

('ok', 'optimal')